# Lab: Multi-Layer Perceptron (MLP)

## 1. Từ Perceptron đến MLP

**Perceptron** (1958) là neuron đơn giản nhất:
$$\hat{y} = \text{sign}(w^T x + b)$$

Nó chỉ học được ranh giới *tuyến tính*. Không thể giải bài XOR (1969 — Minsky & Papert chỉ ra).

**Giải pháp:** xếp chồng nhiều perceptron, thêm hàm phi tuyến giữa các tầng → **Multi-Layer Perceptron**.
$$h_1 = \sigma(W_1 x + b_1)$$
$$h_2 = \sigma(W_2 h_1 + b_2)$$
$$\hat{y} = \text{softmax}(W_3 h_2 + b_3)$$

Theo **Universal Approximation Theorem**, MLP với 1 tầng ẩn đủ rộng + hàm kích hoạt phi tuyến có thể xấp xỉ *bất kỳ* hàm liên tục.

## 2. Hàm kích hoạt

- **ReLU** $\phi(z) = \max(0, z)$: nhanh, gradient ổn, default cho hidden layers.
- **Sigmoid** $\phi(z) = 1/(1+e^{-z})$: $\in (0,1)$, dùng cho output binary.
- **Softmax** $\phi(z)_i = e^{z_i}/\sum_j e^{z_j}$: dùng cho output multiclass.
- **Tanh** $\phi(z) \in (-1,1)$: ít dùng giờ nhưng vẫn ổn.

## 3. Loss và Backpropagation

Với phân loại nhiều lớp (Cross-Entropy Loss):
$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{c=1}^{C} y_{i,c} \log \hat{y}_{i,c}$$

Train bằng **backprop**: tính gradient của $L$ theo từng tham số bằng chain rule, đi ngược từ output về input. Cập nhật tham số bằng gradient descent (hoặc Adam, SGD+momentum...).

**Bẫy phổ biến**: PyTorch's `nn.CrossEntropyLoss` đã *tự* áp dụng softmax bên trong. Nên model trả về **logits**, KHÔNG đặt softmax cuối. Nếu đặt cả hai → softmax(softmax(z)) → loss bị méo, train chậm.

## 1b. Bài toán XOR — bằng chứng cho thấy vì sao PHẢI có tầng ẩn

Perceptron tính $\hat{y} = \mathrm{sign}(w^{\top}x + b)$. Tập các điểm mà nó gán nhãn 1 là một **nửa mặt phẳng** — không hơn. Với XOR, hai điểm lớp 0 là $(0,0), (1,1)$ nằm trên một đường chéo, hai điểm lớp 1 là $(0,1), (1,0)$ nằm trên đường chéo kia. Không nửa mặt phẳng nào chứa đúng một cặp chéo.

**Chứng minh 3 dòng.** Giả sử tồn tại $w_1, w_2, b$ sao cho $w^{\top}x + b > 0$ đúng với hai điểm lớp 1 và $< 0$ với hai điểm lớp 0:

$$
\underbrace{b < 0}_{(0,0)}, \qquad \underbrace{w_1 + w_2 + b < 0}_{(1,1)}, \qquad
\underbrace{w_2 + b > 0}_{(0,1)}, \qquad \underbrace{w_1 + b > 0}_{(1,0)} .
$$

Cộng hai bất đẳng thức cuối: $w_1 + w_2 + 2b > 0$. Từ hai bất đẳng thức đầu: $w_1 + w_2 + b < 0$ và $b < 0$, suy ra $w_1 + w_2 + 2b < 0$. **Mâu thuẫn.** Vậy không có perceptron nào giải được XOR — đúng như Minsky & Papert chỉ ra năm 1969, và chính kết quả này đã đẩy ngành mạng nơ-ron vào "mùa đông AI" đầu tiên.

![XOR trong không gian gốc và trong không gian ẩn](images/01_xor_khong_gian_an.png)

*Hình quan trọng nhất của bài. (a) Trong không gian gốc, mọi đường thẳng đều cắt sai. (b) Nhìn CÙNG 4 điểm đó trong không gian ẩn $(h_1, h_2)$ do tầng ẩn tạo ra: hai điểm lớp 0 bị đẩy trùng lên nhau, và giờ chỉ cần một đường thẳng là tách xong. (c) Đường thẳng đó khi kéo ngược về không gian gốc trở thành một DẢI kẹp giữa hai đường thẳng.*

> **Đây là bản chất của deep learning.** Mạng nơ-ron không "vẽ đường cong phức tạp". Nó học một **phép biến đổi không gian** ($x \mapsto h$) sao cho ở không gian mới, bài toán trở thành tuyến tính — rồi cắt bằng một siêu phẳng. Mọi tầng ẩn đều làm đúng việc đó, tầng sau bẻ tiếp không gian mà tầng trước đã bẻ.
>
> Suy ra: tầng cuối của một mạng phân loại **luôn luôn** chỉ là một bộ phân loại tuyến tính. Phần "thông minh" nằm ở các tầng ẩn — chúng là **bộ trích xuất đặc trưng học được** (learned feature extractor), thay cho việc con người ngồi thiết kế đặc trưng bằng tay.

### Universal Approximation Theorem — phát biểu chính xác và các GIỚI HẠN

**Phát biểu (Cybenko 1989, Hornik 1991).** Cho $\phi$ là hàm kích hoạt liên tục, không đa thức (sigmoid, tanh, ReLU đều thoả). Với mọi hàm liên tục $f$ trên một tập **compact** $K \subset \mathbb{R}^d$ và mọi $\varepsilon > 0$, tồn tại số nguyên $N$ và các tham số $w_i, b_i, v_i$ sao cho

$$
\Big|\, f(x) - \sum_{i=1}^{N} v_i\,\phi(w_i^{\top}x + b_i) \,\Big| < \varepsilon \qquad \forall x \in K .
$$

Nghĩa là: **MLP một tầng ẩn đủ rộng xấp xỉ được mọi hàm liên tục trên miền bị chặn**, với sai số nhỏ tuỳ ý.

Nghe rất mạnh, nhưng phải hiểu **nó KHÔNG nói gì** — đây là chỗ sinh viên hay hiểu sai:

| Định lý NÓI | Định lý KHÔNG nói |
|---|---|
| **Tồn tại** bộ tham số tốt | Rằng gradient descent **tìm được** bộ tham số ấy (bài toán tối ưu không lồi) |
| Sai số xấp xỉ nhỏ tuỳ ý trên tập compact | Bất cứ điều gì về **tổng quát hoá** ra dữ liệu mới (đó là chuyện thống kê, không phải xấp xỉ) |
| Một tầng ẩn là đủ | Rằng một tầng ẩn là **hiệu quả**: $N$ có thể phải lớn theo **cấp số mũ** của $d$ |
| Xấp xỉ hàm liên tục trên miền bị chặn | Điều gì xảy ra **ngoài** miền huấn luyện (ngoại suy — mạng nơ-ron ngoại suy rất tệ) |

**Vì sao vẫn đi SÂU thay vì đi RỘNG?** Có những hàm mà mạng sâu $L$ tầng biểu diễn được với $\mathcal{O}(L)$ neuron, còn mạng 1 tầng ẩn cần $\mathcal{O}(2^L)$ neuron (Telgarsky 2016, Montúfar 2014). Trực giác: mỗi tầng ReLU **gấp** không gian đầu vào lại; $L$ lần gấp tạo ra số vùng tuyến tính tăng theo hàm mũ, trong khi thêm neuron vào cùng một tầng chỉ tăng tuyến tính. Sâu = **tái sử dụng** đặc trưng; rộng = liệt kê đặc trưng.


## 2b. Hàm kích hoạt: điều quan trọng nằm ở ĐẠO HÀM

Khi chọn hàm kích hoạt, đừng nhìn hình dạng của $\phi$ — hãy nhìn $\phi'$, vì **backprop chỉ nhân với $\phi'$**, không bao giờ dùng tới $\phi$ trong đường đi ngược.

![Năm hàm kích hoạt và đạo hàm của chúng](images/03_ham_kich_hoat.png)

*Hàng trên là hàm, hàng dưới là đạo hàm. Vùng hồng nhạt là **vùng bão hoà** — nơi $\phi' \approx 0$, tức là gradient bị nhân với ~0 và tắt ngóm. Sigmoid tệ nhất: đạo hàm cực đại chỉ $0.25$, nên NGAY CẢ ở điểm tốt nhất nó vẫn làm gradient co lại 4 lần mỗi tầng.*

| Hàm | Miền giá trị | $\max \phi'$ | Ưu | Nhược |
|---|---|---|---|---|
| **Sigmoid** | $(0,1)$ | $0.25$ | output đọc được như xác suất | bão hoà 2 đầu; **không zero-centered** → gradient của mọi trọng số cùng một neuron luôn cùng dấu, đường đi zigzag; có $\exp$ nên chậm |
| **Tanh** | $(-1,1)$ | $1$ | zero-centered → hội tụ nhanh hơn sigmoid | vẫn bão hoà hai đầu |
| **ReLU** | $[0,\infty)$ | $1$ | không bão hoà bên phải, tính cực rẻ, tạo activation thưa | **dying ReLU**: neuron rơi vào vùng $z<0$ với mọi dữ liệu thì $\phi'=0$ vĩnh viễn → chết hẳn |
| **LeakyReLU** | $\mathbb{R}$ | $1$ | $\phi'=\alpha$ bên trái nên neuron không chết | thêm 1 siêu tham số $\alpha$ (PyTorch mặc định 0.01, hình dưới vẽ 0.1 cho dễ nhìn) |
| **GELU / SiLU** | $\approx\mathbb{R}$ | $\approx 1.1$ | trơn, đạo hàm liên tục, tốt cho mạng rất sâu | đắt hơn ReLU |

**Mặc định thực dụng: ReLU cho tầng ẩn.** Đổi sang LeakyReLU/GELU khi thấy nhiều neuron chết hoặc khi mạng rất sâu.

### Vanishing / exploding gradient — nguyên nhân toán học

Với mạng $L$ tầng, quy tắc chuỗi cho

$$
\frac{\partial L}{\partial W^{(1)}} \;\propto\; \prod_{l=2}^{L} \Big(\phi'(z^{(l)})\odot W^{(l)}\Big) .
$$

Đây là **tích của $L-1$ số hạng**. Gọi $\kappa$ là độ lớn điển hình của mỗi số hạng:

- $\kappa < 1$ (ví dụ sigmoid với $\phi' \le 0.25$) $\Rightarrow$ gradient $\sim \kappa^{L}$ **tắt theo cấp số nhân** → **vanishing gradient**: tầng gần input gần như đứng yên, mạng chỉ học được vài tầng cuối.
- $\kappa > 1$ (trọng số khởi tạo quá lớn, hoặc RNN chuỗi dài) $\Rightarrow$ gradient **bùng nổ** → loss nhảy lên `NaN`.

![Vanishing gradient trên mạng 10 tầng](images/04_vanishing_gradient.png)

*Trái: đo thực tế trên mạng 10 tầng ẩn cài bằng numpy. Với sigmoid, gradient ở tầng 1 nhỏ hơn tầng 10 khoảng 4 CHỮ SỐ THẬP PHÂN — tầng đầu coi như không được cập nhật. Với ReLU + He init, đường gần như nằm ngang. Phải: bản chất là phép nhân luỹ thừa.*

| Giải pháp | Cách hoạt động |
|---|---|
| **ReLU / LeakyReLU** | $\phi'=1$ ở vùng dương → không co gradient |
| **Khởi tạo He / Xavier** | ép $\mathbb{E}[\kappa] \approx 1$ ngay từ đầu (xem mục 3c) |
| **Batch Normalization** | chuẩn hoá $z$ về mean 0 var 1 mỗi mini-batch → giữ activation khỏi vùng bão hoà, đồng thời làm mặt loss "tròn" hơn |
| **Residual connection** $y = x + F(x)$ | tạo **đường tắt** cho gradient: $\partial y/\partial x = I + \partial F/\partial x$, số hạng $I$ giúp gradient không bị co theo cấp số nhân qua nhiều tầng (nó cộng thêm một đường đi thẳng, không bị nhân với $\phi'$) — chính là chìa khoá của ResNet và Transformer |
| **Gradient clipping** | chặn chuẩn gradient ở ngưỡng $\tau$: `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` — chống **exploding** |

### Bảng chọn hàm kích hoạt CUỐI và hàm mất mát theo loại bài toán

| Bài toán | Số node output | Kích hoạt cuối | Loss (PyTorch) | Ghi chú |
|---|---|---|---|---|
| **Nhị phân** | 1 | *không có* (logit) | `BCEWithLogitsLoss` | hoặc `Sigmoid` + `BCELoss`, nhưng kém ổn định số học hơn |
| **Đa lớp** (1 nhãn/mẫu) | $C$ | *không có* (logits) | `CrossEntropyLoss` | **KHÔNG** đặt `Softmax` ở cuối model |
| **Đa nhãn** (nhiều nhãn/mẫu) | $C$ | *không có* (logits) | `BCEWithLogitsLoss` | mỗi lớp là một bài toán nhị phân độc lập — **không** dùng softmax |
| **Hồi quy** | 1 hoặc $k$ | *không có* (tuyến tính) | `MSELoss` / `L1Loss` / `SmoothL1Loss` | dùng L1/Huber khi có outlier |
| **Hồi quy ràng buộc dương** | 1 | `Softplus` hoặc `exp` | `MSELoss` trên log | ví dụ dự đoán giá, thời lượng |

## 3b. Suy diễn BACKPROPAGATION đầy đủ

### Ký hiệu (đọc kỹ trước khi vào công thức)

| Ký hiệu | Ý nghĩa | Kích thước |
|---|---|---|
| $L$ | tổng số tầng có trọng số | vô hướng |
| $a^{(0)} = x$ | đầu vào | $n_0 \times 1$ |
| $W^{(l)}$ | ma trận trọng số của tầng $l$ | $n_{l-1} \times n_l$ |
| $b^{(l)}$ | bias của tầng $l$ | $n_l \times 1$ |
| $z^{(l)} = W^{(l)\top}a^{(l-1)} + b^{(l)}$ | **pre-activation** (tổng có trọng số, chưa qua phi tuyến) | $n_l \times 1$ |
| $a^{(l)} = \phi(z^{(l)})$ | **activation** của tầng $l$ | $n_l \times 1$ |
| $\mathcal{L}$ | hàm mất mát trên một mẫu | vô hướng |
| $\delta^{(l)} \equiv \dfrac{\partial \mathcal{L}}{\partial z^{(l)}}$ | **"lỗi" của tầng $l$** — đại lượng trung tâm của backprop | $n_l \times 1$ |
| $\odot$ | nhân từng phần tử (Hadamard) | |

Vì sao lại định nghĩa $\delta$ theo $z$ chứ không theo $a$ hay theo $W$? Vì $z$ là **nút thắt cổ chai**: mọi ảnh hưởng của $W^{(l)}$ và $b^{(l)}$ lên loss đều phải đi qua $z^{(l)}$. Biết $\delta^{(l)}$ là biết mọi gradient của tầng đó ngay lập tức.

### Bốn phương trình của backpropagation

**(BP1) Lỗi ở tầng cuối.**
$$
\boxed{\;\delta^{(L)} = \nabla_{a}\mathcal{L} \;\odot\; \phi'\!\big(z^{(L)}\big)\;}
$$
Đọc là: *"loss nhạy với $z^{(L)}$ bao nhiêu"* = *"loss nhạy với output bao nhiêu"* × *"output nhạy với $z^{(L)}$ bao nhiêu"*.

**(BP2) Truyền ngược lỗi.**
$$
\boxed{\;\delta^{(l)} = \Big(W^{(l+1)}\,\delta^{(l+1)}\Big)\;\odot\;\phi'\!\big(z^{(l)}\big)\;}
$$
Hai bước: (i) $W^{(l+1)}\delta^{(l+1)}$ **kéo lỗi từ tầng sau về** — chính ma trận đã dùng lúc đi xuôi, nay dùng ở dạng chuyển vị; (ii) nhân với $\phi'(z^{(l)})$ để **đi ngược qua hàm phi tuyến**. Nhìn vào $\phi'$ ở đây là thấy ngay vanishing gradient sinh ra từ đâu.

**(BP3) Gradient theo bias.**
$$
\boxed{\;\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \delta^{(l)}\;}
$$
Bias cộng thẳng vào $z$ nên đạo hàm bằng đúng $\delta$.

**(BP4) Gradient theo trọng số.**
$$
\boxed{\;\frac{\partial \mathcal{L}}{\partial W^{(l)}} = a^{(l-1)}\,\delta^{(l)\top}\;}
\qquad\text{(ma trận } n_{l-1}\times n_l\text{, đúng bằng kích thước của } W^{(l)})
$$
Từng phần tử: $\partial \mathcal{L}/\partial W^{(l)}_{jk} = a^{(l-1)}_j\,\delta^{(l)}_k$ — *"tín hiệu vào" × "lỗi ra"*. Trọng số chỉ được cập nhật mạnh khi **cả hai** đều lớn: nếu neuron trước im lặng ($a_j = 0$) thì trọng số đó không học được gì trong bước này.

### Vì sao gọi là "back-PROPAGATION"?

Nếu tính gradient một cách ngây thơ bằng quy tắc chuỗi cho từng tham số riêng lẻ, mỗi tham số phải đi lại toàn bộ đường dẫn tới loss — chi phí $\mathcal{O}(P^2)$ với $P$ là số tham số. Backprop **lưu lại** $\delta^{(l)}$ rồi tái sử dụng nó cho toàn bộ trọng số của tầng đó, đưa chi phí xuống $\mathcal{O}(P)$ — xấp xỉ đúng bằng chi phí của một lượt forward. Đó là toàn bộ "phép màu": **quy hoạch động trên đồ thị tính toán**.

![Đồ thị tính toán và luồng backprop](images/05_backprop_do_thi.png)

*Cùng một đồ thị, hai chiều đi: mũi tên liền (xanh) tính giá trị và LƯU LẠI mọi kết quả trung gian; mũi tên đứt (đỏ) đi ngược để tính đạo hàm, dùng lại đúng các giá trị vừa lưu. Vì phải lưu, huấn luyện tốn bộ nhớ hơn suy luận nhiều lần — đó cũng là lý do `torch.no_grad()` giúp tiết kiệm RAM khi đánh giá.*

### Trường hợp đặc biệt tuyệt đẹp: softmax + cross-entropy

Với output softmax $\hat{y}_c = e^{z_c}/\sum_j e^{z_j}$ và loss $\mathcal{L} = -\sum_c y_c \log \hat{y}_c$, ta có

$$
\frac{\partial \mathcal{L}}{\partial z_c} = \hat{y}_c - y_c
\qquad\Longleftrightarrow\qquad
\delta^{(L)} = \hat{y} - y .
$$

Không còn $\phi'$ nào cả! Đạo hàm của softmax triệt tiêu đúng mẫu số của log. Hệ quả cực kỳ quan trọng: **khi mạng dự đoán sai bét, gradient LỚN** (bằng $\hat y - y$, gần 1). Nếu thay bằng MSE + sigmoid thì $\delta = (\hat y - y)\,\sigma'(z)$, mà $\sigma'\approx 0$ khi sai bét → **sai càng nặng, học càng chậm**. Đó là lý do bài toán phân loại luôn dùng cross-entropy chứ không dùng MSE.

### Vì sao `nn.CrossEntropyLoss` nhận LOGITS chứ không nhận xác suất

Tính softmax rồi mới lấy log là **không an toàn về số học**. Với $z = [1000, 1001]$, $e^{1000}$ tràn số (`inf`) trong `float32` (ngưỡng tràn $\approx e^{88}$), cho ra `nan`. Với $z = [-1000, -1001]$ thì $e^{-1000}$ về $0$, `log(0) = -inf`.

**Log-sum-exp trick.** Với $m = \max_j z_j$:

$$
\log \sum_j e^{z_j}
= \log \sum_j e^{\,z_j - m + m}
= m + \log \sum_j e^{\,z_j - m} .
$$

Sau khi trừ $m$, số mũ lớn nhất bằng $0$ nên $e^{z_j-m} \in (0, 1]$ — **không bao giờ tràn**. Từ đó

$$
\log \hat{y}_c = z_c - \log\sum_j e^{z_j} = z_c - m - \log\sum_j e^{\,z_j - m},
$$

và cross-entropy tính trực tiếp theo công thức này, **không hề tạo ra xác suất trung gian**. Muốn dùng được mẹo đó, hàm loss phải nhận được $z$ (logits) — nếu bạn đã tự bấm `softmax` trong `forward` thì thông tin ấy đã mất, và PyTorch buộc phải làm `log(softmax(softmax(z)))`: vừa sai về toán, vừa mất ổn định số học, vừa làm gradient bé đi.

> **Quy tắc nhớ:** `nn.CrossEntropyLoss = LogSoftmax + NLLLoss`. Model trả **logits**. Chỉ gọi `softmax` khi bạn thực sự cần con số xác suất để hiển thị, và gọi nó **ngoài** hàm loss.

Ô code dưới đây cài lại toàn bộ (BP1)–(BP4) bằng numpy thuần và **kiểm tra gradient** bằng sai phân hữu hạn — cách chuẩn để biết mình viết backward đúng hay sai.


In [ ]:
# MLP tối giản bằng NUMPY THUẦN — để thấy rõ backprop, không có framework nào che
# (chạy được kể cả khi máy không cài PyTorch)
import numpy as np
from sklearn.datasets import make_moons

def relu(z):      return np.maximum(0, z)
def d_relu(z):    return (z > 0).astype(float)
def sigmoid(z):   return 1 / (1 + np.exp(-np.clip(z, -60, 60)))

class TinyMLP:
    """Kiến trúc [d_in, h, 1]: Linear → ReLU → Linear → Sigmoid, loss = BCE."""
    def __init__(self, d_in, h, seed=0):
        rng = np.random.default_rng(seed)
        # He init cho tầng ReLU, Xavier cho tầng output
        self.W1 = rng.normal(0, np.sqrt(2 / d_in), (d_in, h)); self.b1 = np.zeros(h)
        self.W2 = rng.normal(0, np.sqrt(1 / h),    (h, 1));    self.b2 = np.zeros(1)

    def forward(self, X):
        self.X  = X
        self.z1 = X @ self.W1 + self.b1      # (n, h)
        self.a1 = relu(self.z1)              # (n, h)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.p  = sigmoid(self.z2)           # (n, 1)
        return self.p

    def loss(self, y):
        p = np.clip(self.p.ravel(), 1e-9, 1 - 1e-9)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

    def backward(self, y):
        n = len(y)
        # sigmoid + BCE rút gọn: delta của tầng output = (p - y)
        d2 = (self.p - y.reshape(-1, 1)) / n          # (n, 1)
        gW2 = self.a1.T @ d2                          # a^(l-1)^T · delta
        gb2 = d2.sum(0)
        # truyền ngược: delta^(l) = (W^(l+1) delta^(l+1)) ⊙ phi'(z^(l))
        d1 = (d2 @ self.W2.T) * d_relu(self.z1)       # (n, h)
        gW1 = self.X.T @ d1
        gb1 = d1.sum(0)
        return gW1, gb1, gW2, gb2

    def step(self, y, lr):
        gW1, gb1, gW2, gb2 = self.backward(y)
        self.W1 -= lr * gW1; self.b1 -= lr * gb1
        self.W2 -= lr * gW2; self.b2 -= lr * gb2

X, y = make_moons(n_samples=400, noise=0.2, random_state=0)
X = (X - X.mean(0)) / X.std(0); y = y.astype(float)

net = TinyMLP(2, 32, seed=0)

# --- KIỂM TRA GRADIENT (gradient checking): so đạo hàm giải tích với sai phân hữu hạn
net.forward(X); gW1, _, _, _ = net.backward(y)
eps, i, j = 1e-5, 1, 7
net.W1[i, j] += eps; net.forward(X); lp = net.loss(y)
net.W1[i, j] -= 2 * eps; net.forward(X); lm = net.loss(y)
net.W1[i, j] += eps; net.forward(X)
print(f'grad giải tích  = {gW1[i, j]: .8f}')
print(f'grad sai phân   = {(lp - lm) / (2 * eps): .8f}   ← trùng nhau ⇒ backprop viết đúng\n')

# --- Huấn luyện
for epoch in range(2001):
    net.forward(X)
    if epoch % 400 == 0:
        acc = ((net.p.ravel() > .5) == (y > .5)).mean()
        print(f'epoch {epoch:5d}   loss = {net.loss(y):.4f}   acc = {acc * 100:.1f}%')
    net.step(y, lr=0.5)

n_params = net.W1.size + net.b1.size + net.W2.size + net.b2.size
print(f'\nTổng tham số: {n_params}  (= 2·32 + 32 + 32·1 + 1)')


## 3c. Khởi tạo trọng số — quyết định mạng có học được hay không

### Vì sao KHÔNG được khởi tạo toàn bộ trọng số bằng 0?

Giả sử $W^{(1)} = 0$ và $b^{(1)} = 0$. Khi đó **mọi neuron trong cùng một tầng có cùng $z$, cùng $a$**. Ở lượt backward, theo (BP4) gradient của chúng cũng **bằng nhau**. Sau bước cập nhật, chúng vẫn bằng nhau. Quy nạp: chúng bằng nhau **mãi mãi**.

$$
\text{Tầng } h \text{ neuron giống hệt nhau} \;\equiv\; \text{tầng chỉ có 1 neuron.}
$$

Đây gọi là **vấn đề đối xứng (symmetry breaking)**: khởi tạo phải **ngẫu nhiên** để mỗi neuron xuất phát ở một chỗ khác nhau và học được đặc trưng khác nhau. (Bias thì khởi tạo 0 vô tư — chỉ cần một trong hai phá đối xứng là đủ, và $W$ đã làm việc đó.)

### Nhưng ngẫu nhiên với PHƯƠNG SAI bao nhiêu?

Đây mới là phần tinh tế. Xét $z = \sum_{i=1}^{n_{in}} w_i x_i$ với các $w_i$ độc lập, trung bình 0:

$$
\mathrm{Var}(z) = n_{in}\,\mathrm{Var}(w)\,\mathrm{Var}(x) .
$$

Muốn tín hiệu **không co lại cũng không phình ra** khi đi qua tầng, ta cần $\mathrm{Var}(z) = \mathrm{Var}(x)$, tức là

$$
\boxed{\;\mathrm{Var}(w) = \frac{1}{n_{in}}\;} \qquad \textbf{Xavier / Glorot} \;-\; \text{cho tanh, sigmoid}
$$

(Bản Glorot đối xứng dùng $\mathrm{Var}(w) = 2/(n_{in}+n_{out})$ để cân bằng cả lượt đi lẫn lượt về.)

Với **ReLU** thì khác: ReLU **vứt bỏ một nửa** tín hiệu (mọi giá trị âm về 0), làm phương sai bị chia đôi. Phải bù lại đúng hệ số 2:

$$
\boxed{\;\mathrm{Var}(w) = \frac{2}{n_{in}}\;} \qquad \textbf{He} \;-\; \text{cho ReLU, LeakyReLU}
$$

![Phân phối activation theo tầng với ba cách khởi tạo](images/10_khoi_tao_trong_so.png)

*Thí nghiệm kinh điển: đẩy dữ liệu qua 6 tầng tanh 500 nút. **Quá nhỏ** ($\sigma=0.01$): activation co về 0 theo cấp số nhân, tới tầng 4 thì std chỉ còn 0.002 — tín hiệu chết, gradient (tỉ lệ với activation theo BP4) cũng chết. **Quá lớn** ($\sigma=1$): mọi giá trị dán chặt vào $\pm 1$, tanh bão hoà hoàn toàn, $\phi'\approx 0$ — gradient chết theo kiểu khác. **Xavier**: activation trải đều qua cả 6 tầng.*

Trong PyTorch, `nn.Linear` mặc định gọi `kaiming_uniform_(a=math.sqrt(5))`, cho phương sai hiệu dụng khoảng $1/(3 n_{in})$, tức gần LeCun hơn là He nên **đa số trường hợp bạn không cần làm gì**. Khi cần chỉ định rõ:

```python
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')  # He
        nn.init.zeros_(m.bias)
```

# THỰC HÀNH: MLP phân loại FashionMNIST

FashionMNIST: 70.000 ảnh 28×28 trắng đen của 10 loại đồ thời trang (áo, quần, giày, túi...). Khó hơn MNIST một chút.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])
trainset = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
testset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(testset,  batch_size=128, shuffle=False, num_workers=0)

class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print(f'Train: {len(trainset)},  Test: {len(testset)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.flat, np.random.choice(len(trainset), 10, replace=False)):
    img, label = trainset[i]
    ax.imshow(img.squeeze() * 0.3530 + 0.2860, cmap='gray')
    ax.set_title(class_names[label]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {n_params:,}')

## 4. Đếm tham số — và vì sao CNN thắng MLP trên ảnh

![Kiến trúc MLP 784-256-128-10 với kích thước ma trận trọng số](images/02_kien_truc_mang.png)

*Mỗi tầng `Linear(n_in → n_out)` có đúng $n_{in}\,n_{out}$ trọng số + $n_{out}$ bias. Riêng tầng đầu tiên đã chiếm 200 960 / 235 146 ≈ **85%** tổng tham số — chỉ vì nó phải nối 784 pixel với 256 neuron.*

| Tầng | Phép tính | Trọng số | Bias | Tổng |
|---|---|---|---|---|
| `Linear(784, 256)` | $784\times256$ | 200 704 | 256 | **200 960** |
| `Linear(256, 128)` | $256\times128$ | 32 768 | 128 | **32 896** |
| `Linear(128, 10)` | $128\times10$ | 1 280 | 10 | **1 290** |
| | | | | **235 146** |

Đối chiếu với `print(f'Total parameters: {n_params:,}')` ở ô trên — con số phải khớp chính xác.

### So sánh với CNN cho cùng bài toán

| | **MLP** (784→256→128→10) | **CNN nhỏ** (2 khối conv + 1 FC) |
|---|---|---|
| Tham số | ~235 000 | ~35 000 |
| Test accuracy trên FashionMNIST | ~88–89% | ~91–93% |
| Xử lý ảnh dịch đi 2 pixel | **hỏng** — mọi pixel là một feature riêng biệt | gần như không đổi |

**Ba lý do CNN thắng:**

1. **Chia sẻ trọng số (weight sharing).** Một filter $3\times3$ chỉ có 9 tham số nhưng được **trượt** qua toàn bộ ảnh. MLP thì mỗi vị trí pixel có bộ trọng số riêng → số tham số bùng nổ và mỗi tham số chỉ nhìn thấy rất ít dữ liệu.
2. **Bất biến tịnh tiến (translation equivariance).** "Cạnh dọc" là cạnh dọc dù nó nằm ở góc trên trái hay góc dưới phải. CNN học một lần rồi dùng ở mọi nơi. MLP phải học lại "cạnh dọc" cho **từng vị trí**, tức là cần nhiều dữ liệu hơn gấp bội để đạt cùng độ chính xác.
3. **Khai thác cấu trúc không gian.** `nn.Flatten()` biến ma trận $28\times28$ thành vector 784 chiều — thông tin *"pixel 5 và pixel 6 nằm cạnh nhau, còn pixel 5 và pixel 33 nằm trên–dưới nhau"* **bị xoá sạch**. Thật vậy: nếu bạn hoán vị ngẫu nhiên 784 pixel theo **cùng một hoán vị cố định** cho mọi ảnh, MLP đạt **đúng cùng độ chính xác**, còn CNN thì sụp đổ. Đó là bằng chứng đơn giản nhất cho thấy MLP không hề "nhìn" ảnh như ảnh.

> **Bài học tổng quát:** kiến trúc mạng chính là cách ta **nhúng tri thức tiên nghiệm (inductive bias)** về dữ liệu vào model. Ảnh → CNN (cấu trúc lưới, cục bộ). Chuỗi/văn bản → RNN, Transformer (thứ tự, phụ thuộc xa). Bảng dữ liệu không có cấu trúc gì đặc biệt → MLP (và thường thua Gradient Boosting).


In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)
    return loss_sum / total, correct / total

num_epochs = 10
train_loss_h, train_acc_h, test_loss_h, test_acc_h = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    running, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total   += y.size(0)

    tr_loss, tr_acc = running / total, correct / total
    te_loss, te_acc = evaluate(model, test_loader)
    train_loss_h.append(tr_loss); train_acc_h.append(tr_acc)
    test_loss_h.append(te_loss);  test_acc_h.append(te_acc)
    print(f'Epoch {epoch+1:2d}/{num_epochs}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc*100:.2f}%  '
          f'test_loss={te_loss:.4f}  test_acc={te_acc*100:.2f}%')

## 5. Optimizer, learning rate và batch size

### 5.1. Bốn optimizer cần biết

Ký hiệu: $g_t = \nabla_\theta \mathcal{L}$ là gradient tại bước $t$, $\eta$ là learning rate.

| Optimizer | Công thức cập nhật | Trực giác |
|---|---|---|
| **SGD** | $\theta_{t+1} = \theta_t - \eta\,g_t$ | đi thẳng theo dốc hiện tại; đơn giản nhất, hay bị zigzag trong thung lũng hẹp |
| **SGD + Momentum** | $v_t = \beta v_{t-1} + g_t$; $\theta_{t+1} = \theta_t - \eta\,v_t$ ($\beta \approx 0.9$) | như hòn bi có **quán tính**: các hướng dao động triệt tiêu nhau, hướng nhất quán được cộng dồn → vượt qua "cao nguyên" và giảm zigzag |
| **RMSProp** | $s_t = \rho s_{t-1} + (1-\rho)g_t^2$; $\theta_{t+1} = \theta_t - \dfrac{\eta}{\sqrt{s_t}+\epsilon} g_t$ | **learning rate riêng cho từng tham số**: tham số nào gradient hay lớn thì bị chia nhiều → bước ngắn lại; tham số ít được cập nhật thì bước dài ra |
| **Adam** | $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$; $s_t = \beta_2 s_{t-1} + (1-\beta_2)g_t^2$; hiệu chỉnh chệch $\hat m_t = m_t/(1-\beta_1^t)$, $\hat s_t = s_t/(1-\beta_2^t)$; $\theta_{t+1} = \theta_t - \dfrac{\eta}{\sqrt{\hat s_t}+\epsilon}\hat m_t$ | **Momentum + RMSProp**, cộng thêm hiệu chỉnh chệch cho các bước đầu (vì $m_0=s_0=0$ nên ước lượng ban đầu bị kéo về 0) |

| Khi nào dùng | Optimizer | Learning rate mặc định |
|---|---|---|
| Muốn chạy được ngay, ít phải chỉnh | **Adam** / **AdamW** | `1e-3` (Adam), `3e-4` cho Transformer |
| Muốn kết quả cuối tốt nhất, có thời gian tune (thị giác máy tính, ResNet) | **SGD + momentum 0.9** + lr schedule | `0.1` với batch 256, giảm 10 lần ở các mốc |
| Mạng có embedding/gradient thưa (NLP) | **Adam** | `1e-3` |
| Cần weight decay đúng nghĩa L2 | **AdamW** | `1e-3`, `weight_decay=0.01` |

> **Ghi chú quan trọng:** trong `torch.optim.Adam`, tham số `weight_decay` được cộng thẳng vào gradient nên nó **bị chia** cho $\sqrt{\hat s_t}$ — không còn tương đương với phạt L2 nữa. `AdamW` tách weight decay ra khỏi bước adaptive, và đó là lý do AdamW là mặc định trong mọi model hiện đại.

### 5.2. Learning rate — siêu tham số quan trọng nhất

![Ảnh hưởng của learning rate lên đường loss](images/08_learning_rate.png)

*Cùng mạng, cùng khởi tạo, chỉ đổi `lr`. Quá nhỏ: 300 epoch vẫn chưa học xong. Vừa: mượt và nhanh. Hơi lớn: vẫn xuống nhưng nảy lên nảy xuống. Quá lớn: loss bắn lên rồi kẹt cứng ở 50% accuracy (bằng đoán bừa) — với ReLU điều này thường có nghĩa là **toàn bộ neuron đã chết** sau một bước cập nhật khổng lồ.*

Quy trình thực dụng: thử `lr` theo thang $\times 3$ (`1e-4, 3e-4, 1e-3, 3e-3, 1e-2`), chạy 1–2 epoch mỗi giá trị, chọn cái cho loss giảm nhanh nhất mà không nảy. Sau đó dùng **learning rate schedule** (`StepLR`, `CosineAnnealingLR`, `ReduceLROnPlateau`) để giảm dần về cuối — bước lớn lúc đầu để đi nhanh, bước nhỏ lúc sau để tinh chỉnh.

### 5.3. Batch size — đánh đổi giữa nhiễu, tốc độ và tổng quát hoá

| | **Full-batch** ($B = N$) | **Mini-batch** ($B = 32..512$) | **SGD thuần** ($B=1$) |
|---|---|---|---|
| Gradient | chính xác | ước lượng nhiễu | rất nhiễu |
| Đường loss | mượt, đơn điệu | có dao động nhỏ | dao động mạnh |
| Số bước cập nhật / epoch | 1 | $N/B$ | $N$ |
| Tận dụng GPU | tốt nhưng chỉ 1 bước | **tốt nhất** | rất kém (không song song hoá được) |
| Bộ nhớ | $\mathcal{O}(N)$ — thường không vừa | $\mathcal{O}(B)$ | $\mathcal{O}(1)$ |
| Khả năng thoát điểm yên ngựa / cực tiểu xấu | kém — kẹt là kẹt luôn | **tốt** (nhiễu đóng vai trò regularizer) | tốt nhưng phí thời gian |

**Nhiễu của mini-batch là một TÍNH NĂNG, không phải lỗi**: nó giúp nhảy ra khỏi cực tiểu địa phương hẹp và có xu hướng dẫn tới cực tiểu "phẳng" — loại cực tiểu thường tổng quát hoá tốt hơn. Batch quá lớn (vài nghìn) hay bị mất phần nhiễu này và test accuracy giảm, trừ khi tăng lr theo (quy tắc kinh nghiệm: **nhân đôi batch thì nhân đôi lr**, kèm warm-up).

Trong lab này `batch_size=128` với 60 000 ảnh → $60000/128 \approx 469$ bước cập nhật mỗi epoch.


In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, train_loss_h, 'o-', label='Train')
axes[0].plot(epochs, test_loss_h,  's-', label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [a*100 for a in train_acc_h], 'o-', label='Train')
axes[1].plot(epochs, [a*100 for a in test_acc_h],  's-', label='Test')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Sức chứa của mạng, overfitting và regularization

### 6.1. Mạng học như thế nào qua thời gian

![Ranh giới quyết định tiến hoá theo epoch](images/06_ranh_gioi_theo_epoch.png)

*Cùng một mạng 2–16–16–1 (numpy thuần) trên `make_moons`. Epoch 0 (chỉ mới khởi tạo ngẫu nhiên): ranh giới gần như thẳng. Epoch 20: đã bắt đầu cong. Epoch 1000: ôm sát hình lưỡi liềm. Mạng đi từ model đơn giản → model phức tạp một cách **tự nhiên theo thời gian huấn luyện** — đây chính là lý do sâu xa khiến `early stopping` có tác dụng như một dạng regularization.*

### 6.2. Nhiều neuron ẩn hơn = sức chứa lớn hơn = dễ overfit hơn

![Ảnh hưởng của số neuron ẩn](images/07_so_neuron_an.png)

*Với 1 neuron ẩn, mạng chỉ vẽ nổi 1 đường thẳng → **underfit** (bias cao). Với 64 neuron trên vỏn vẹn 120 điểm train nhiễu, ranh giới lượn theo từng điểm nhiễu, sinh cả những "hòn đảo" cô lập → **overfit** (variance cao): train accuracy 98% nhưng test chỉ 85%.*

Lưu ý điều thú vị mà đồ thị bên phải cho thấy: **train loss giảm dần theo số neuron (chỉ nhích nhẹ ở 64–128 neuron vì 4000 epoch GD chưa đủ cho mạng rộng hội tụ hẳn), còn test loss giảm rồi TĂNG gấp 2–3 lần**. Đây đúng là đường cong bias–variance đã gặp ở Lab 01 với bậc đa thức — chỉ đổi trục hoành.

### 6.3. Overfitting và bộ công cụ chống lại nó

![Early stopping và dropout](images/09_overfitting_dropout.png)

*Trái: dấu hiệu overfitting kinh điển — train loss tiến về 0 trong khi validation loss chạm đáy rồi đi lên. Điểm đáy đó chính là chỗ nên dừng. Phải: dropout tắt ngẫu nhiên một phần neuron ở mỗi bước huấn luyện.*

| Kỹ thuật | Làm gì | Dùng trong PyTorch | Lưu ý |
|---|---|---|---|
| **Weight decay (L2)** | thêm $\lambda\,\theta^{\top}\theta$ vào loss → kéo trọng số về 0, model "mượt" hơn | `optim.AdamW(..., weight_decay=1e-2)` | với Adam hãy dùng **AdamW** |
| **Dropout** | mỗi bước tắt ngẫu nhiên tỉ lệ $p$ neuron | `nn.Dropout(0.3)` sau mỗi ReLU | **bắt buộc** `model.eval()` khi đánh giá |
| **Early stopping** | dừng ở epoch có val loss nhỏ nhất | tự theo dõi, lưu `state_dict` tốt nhất | cần tập validation **riêng**, không dùng test |
| **Data augmentation** | sinh thêm dữ liệu bằng phép biến đổi giữ nhãn | `transforms.RandomHorizontalFlip()`, `RandomCrop` | chỉ áp cho **train**, không áp cho test |
| **Batch Normalization** | chuẩn hoá $z$ theo mini-batch | `nn.BatchNorm1d(256)` | cũng có tác dụng regularize nhẹ; cũng đổi hành vi giữa train/eval |
| **Giảm sức chứa** | bớt neuron / bớt tầng | — | cách rẻ nhất, thử trước tiên |
| **Thêm dữ liệu** | — | — | luôn là cách tốt nhất nếu khả thi |

**Về dropout và việc scale khi inference.** Lúc train, mỗi neuron chỉ "có mặt" với xác suất $1-p$, nên kỳ vọng đầu vào của tầng sau là $(1-p)\times$ giá trị đầy đủ. Lúc inference bật lại tất cả thì đầu vào bỗng lớn hơn $1/(1-p)$ lần → thống kê lệch, kết quả sai. Có hai cách bù:

- **Dropout cổ điển**: lúc inference nhân activation với $(1-p)$.
- **Inverted dropout** (PyTorch và mọi framework hiện đại dùng cách này): lúc **train** đã chia sẵn cho $(1-p)$, nên lúc inference **không phải làm gì** — chỉ cần tắt dropout đi.

Việc "tắt dropout đi" chính là điều `model.eval()` làm (đồng thời chuyển BatchNorm sang dùng thống kê chạy tích luỹ thay vì thống kê của batch hiện tại). Quên gọi nó là một trong những lỗi phổ biến nhất của người mới — và nó **không báo lỗi**, chỉ làm kết quả đánh giá tệ đi và nhiễu ngẫu nhiên giữa các lần chạy.


In [ ]:
model.eval()
cm = torch.zeros(10, 10, dtype=torch.long)
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, preds):
            cm[t.item(), p.item()] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm.numpy(), cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix — MLP trên FashionMNIST')
for i in range(10):
    for j in range(10):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

off_diag = cm.clone(); off_diag.fill_diagonal_(0)
i_max, j_max = np.unravel_index(off_diag.numpy().argmax(), off_diag.shape)
print(f'Cặp nhầm nhiều nhất: "{class_names[i_max]}" → "{class_names[j_max]}" '
      f'({cm[i_max, j_max]} lần) — hai loại trông giống nhau.')

## 7. Bẫy thường gặp — checklist trước khi chạy

| # | Bẫy | Triệu chứng | Cách sửa |
|---|---|---|---|
| 1 | **Quên `optimizer.zero_grad()`** | loss giảm rồi bất ngờ tăng vọt / `NaN` | PyTorch **cộng dồn** gradient vào `.grad`. Không xoá thì gradient của mọi batch trước đó bị cộng lại. Luôn `zero_grad()` **trước** `loss.backward()` |
| 2 | **Quên `model.eval()`** khi đánh giá | test accuracy thấp hơn thực tế và **đổi giữa các lần chạy** | `model.eval()` trước vòng đánh giá, `model.train()` khi quay lại huấn luyện. Ảnh hưởng tới Dropout và BatchNorm |
| 3 | **Quên `torch.no_grad()`** khi đánh giá | RAM/VRAM phình to, chậm | bọc vòng đánh giá trong `with torch.no_grad():` |
| 4 | **Đặt `Softmax` ở cuối model** khi dùng `CrossEntropyLoss` | loss giảm rất chậm, accuracy kém | bỏ softmax đi, trả **logits** (xem mục 3b) |
| 5 | **Quên chuẩn hoá input** | loss giảm chậm hoặc kẹt | `transforms.Normalize(mean, std)`; ảnh 0–255 phải đưa về ~$[-1,1]$ hoặc $[0,1]$ |
| 6 | **Learning rate quá lớn** | loss `NaN`, hoặc kẹt ở accuracy của đoán bừa | giảm 3–10 lần; kiểm tra bằng cách chạy 1 epoch |
| 7 | **Dùng test set để chọn siêu tham số** | test accuracy đẹp trên báo cáo, thất bại khi triển khai | tách **validation** riêng; test chỉ đụng vào **một lần cuối** |
| 8 | **Quên `.to(device)` cho dữ liệu** | `RuntimeError: Expected all tensors to be on the same device` | `x, y = x.to(device), y.to(device)` |
| 9 | **Nhầm `loss.item()` với `loss`** | giữ nguyên cả đồ thị tính toán → rò rỉ bộ nhớ | cộng dồn thống kê bằng `loss.item()` |
| 10 | **Không cố định seed** | không tái lập được kết quả, không so sánh được 2 lần chạy | `torch.manual_seed`, `np.random.seed` (và ghi rõ trong báo cáo) |
| 11 | **Shuffle tập test** | không sai kết quả nhưng khiến việc debug rối | `shuffle=True` cho train, `False` cho test |
| 12 | **Model không học gì (loss đứng im ngay từ đầu)** | — | kiểm tra theo thứ tự: lr, chuẩn hoá input, nhãn có đúng dtype `long` không, model có thực sự nhận gradient không (`p.grad is None`?) |

### Mẹo debug số 1: THỬ OVERFIT 1 BATCH

Trước khi huấn luyện thật, lấy **đúng một batch nhỏ** (ví dụ 8 mẫu) và train vài trăm bước. Nếu code đúng, mạng **phải** đạt loss ≈ 0 và accuracy 100% trên chính 8 mẫu đó — vì nó thừa sức học thuộc lòng. Nếu không được, lỗi nằm ở **code** (nhãn sai, quên `zero_grad`, lr sai, model đứt kết nối gradient) chứ không phải ở dữ liệu hay kiến trúc. Mẹo này tiết kiệm hàng giờ đồng hồ.


## Tổng kết

1. MLP = nhiều tầng Linear + phi tuyến (ReLU). Học được ranh giới phi tuyến tuỳ ý nếu đủ lớn.
2. **Output là logits**, KHÔNG softmax cuối khi dùng `CrossEntropyLoss`.
3. Loss xuống, accuracy lên — nếu test loss bắt đầu tăng trong khi train loss vẫn giảm → **overfitting**.
4. **Confusion matrix** chỉ ra cặp lớp dễ nhầm — gợi ý cải tiến model hoặc dữ liệu.

## Khi nào MLP đủ — khi nào cần CNN?
MLP "flatten" ảnh thành vector → vứt mất thông tin không gian (pixel cạnh nhau). Với ảnh, **CNN** thường vượt MLP với ít tham số hơn nhiều.

# BÀI TẬP VỀ NHÀ

## Bài 1: Sâu hơn có tốt hơn không?
Thử 3 kiến trúc và so sánh test acc sau 10 epoch:
1. `784 → 128 → 10`
2. `784 → 256 → 128 → 10` (như lab)
3. `784 → 512 → 256 → 128 → 10`

Có hiện tượng diminishing returns (thêm tầng không giúp nhiều) không?

## Bài 2: Dropout chống overfitting
Thêm `nn.Dropout(0.3)` sau mỗi `nn.ReLU()`. Train 20 epoch. So sánh khoảng cách `train_acc - test_acc` trước và sau khi thêm Dropout. Dropout có giúp giảm gap không?

*Gợi ý:* Dropout chỉ active khi `model.train()`.

## Bài 3: Optimizer comparison
Train cùng kiến trúc với 3 optimizer:
- `optim.SGD(lr=0.01)`
- `optim.SGD(lr=0.01, momentum=0.9)`
- `optim.Adam(lr=1e-3)` (hiện tại)

Vẽ 3 đường loss trên cùng đồ thị. Cái nào hội tụ nhanh nhất?

## Bài 4: Áp dụng cho MNIST
Đổi `datasets.FashionMNIST` thành `datasets.MNIST`. Đổi normalize mean/std (`(0.1307,), (0.3081,)`). Train 10 epoch. So với FashionMNIST: MNIST dễ hơn (>97%) hay khó hơn?

## Bài 5: CIFAR-10 (nâng cao)
CIFAR-10: ảnh màu 32×32, 10 lớp đối tượng.
1. `datasets.CIFAR10(root='./data', ...)`.
2. Mean/std: `(0.4914, 0.4822, 0.4465)`, `(0.2470, 0.2435, 0.2616)`.
3. Input dim = 3 × 32 × 32 = 3072.
4. Train 15 epoch.

Quan sát: MLP trên CIFAR-10 thường chỉ đạt ~50% — kém xa CNN (~85%+). Vì sao? *MLP flatten ảnh, mất thông tin không gian. CNN dùng convolutional filter → khai thác được spatial structure.*